# Module 4: Explainable AI (XAI) with SHAP & Business Decision Playbooks
### Course: Applied Data Science & Machine Learning in Production
**Case Study:** Customer Churn & Lifetime Value (LTV) Prediction for a Subscription Business

---
### The Business Challenge:
A predictive model that outputs `P(churn) = 0.84` is useless to a Customer Success Manager unless they know **why** the customer is at risk. 
- Is the customer struggling with product bugs? (Support tickets)
- Are they disengaging? (Usage drop)
- Is the pricing model unfavorable? (Monthly charges vs contract type)

Without Explainability, retention teams apply blunt, expensive discounts to everyone. With **Explainable AI (XAI)**, we map individualized feature attributions directly to targeted **Retention Playbooks**.

### Learning Objectives:
1. **Cooperative Game Theory:** Understand Shapley Values and TreeSHAP properties (Efficiency, Symmetry, Additivity).
2. **Global Model Interpretability:** Generate SHAP Beeswarm and summary bar charts to rank overall feature importance.
3. **Local Customer Explanations:** Build Waterfall plots dissecting individual customer churn drivers.
4. **Actionable Business Decision Matrix:** Connect SHAP attributions to automated operational workflows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import shap

from sklearn.ensemble import GradientBoostingClassifier
import sys
sys.path.append("..")
from src.features import create_preprocessor
from src.data_loader import load_raw_data, temporal_train_test_split

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/raw/subscriptions_raw.csv")
df = load_raw_data(DATA_PATH)
train_df, test_df = temporal_train_test_split(df, cutoff_quantile=0.80)

feature_cols = [c for c in train_df.columns if c not in ["customer_id", "signup_date", "churn", "customer_ltv"]]
X_train = train_df[feature_cols]
y_train = train_df["churn"].values
X_test = test_df[feature_cols]
y_test = test_df["churn"].values

print(f"Dataset prepared: {X_train.shape[0]} train rows, {X_test.shape[0]} test rows.")

## 1. Model Training & SHAP TreeExplainer Initialization
We train our baseline model and initialize the SHAP `TreeExplainer` on transformed feature vectors.

In [ ]:
preprocessor = create_preprocessor()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Retrieve transformed feature names
cat_encoder = preprocessor.named_steps["preprocessor"].named_transformers_["cat"]
cat_names = cat_encoder.get_feature_names_out(["contract_type", "payment_method", "internet_service"]).tolist()
num_names = [
    "tenure_months", "monthly_charges", "total_charges",
    "num_support_tickets", "avg_weekly_logins", "usage_drop_pct_last_30d",
    "charge_to_tenure_ratio", "tickets_per_tenure", "engagement_index", "expected_spend_diff"
]
bin_names = ["has_premium_support", "multi_seat_license"]
feature_names = num_names + cat_names + bin_names

model = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42)
model.fit(X_train_proc, y_train)

# Initialize TreeExplainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_proc)
print("TreeExplainer successfully initialized. SHAP values computed for test set.")

## 2. Global Interpretability: SHAP Beeswarm Plot
A beeswarm plot reveals not only which features are important, but the direction of their impact on churn risk.

In [ ]:
# TODO: Student Exercise 1
# Generate a SHAP summary plot (beeswarm) for 'shap_values' and 'X_test_proc'.
# Pass 'feature_names' to label each row cleanly.

# shap.summary_plot(...)

## 3. Local Customer Explanations: Waterfall Plot
Let's zoom into a high-risk customer (e.g. index 12 in the test set) and dissect the positive and negative push on their log-odds of churning.

In [ ]:
# TODO: Student Exercise 2
# Construct a shap.Explanation object for a single customer record.
# Plot a waterfall plot displaying the top 8 contributors.

# explanation_obj = shap.Explanation(...)
# shap.waterfall_plot(...)

## 4. Translating XAI into Business Retention Playbooks
Instead of giving a blanket discount, we programmatically route customers based on their primary SHAP risk driver:

| Primary Positive SHAP Driver | Diagnostic Interpretation | Automated Retention Playbook |
|---|---|---|
| **`usage_drop_pct_last_30d`** | Customer is abandoning platform workflow | Automated in-app product re-engagement tour + CSM email |
| **`num_support_tickets`** | Product friction / unresolved bugs | Immediate escalation call with Senior Customer Support |
| **`contract_type_Month-to-month`** | Lack of contractual lock-in | 15% discount for upgrading to annual contract |
| **`monthly_charges`** | Perceived price-to-value deficit | Personalized bundle restructure |

In [ ]:
# TODO: Student Exercise 3
# Implement a function that takes a customer's SHAP values and feature names,
# identifies the single highest positive contributor to churn,
# and returns the corresponding business intervention playbook.

## 5. Summary & Key MLOps Takeaways
- **Interpretability builds trust:** Regulators, business executives, and customer success teams need causal reasoning, not black boxes.
- **Explainability optimizes budgets:** Targeted playbooks save money compared to indiscriminate blanket discounts.
- Next module: **Post-Deployment Data Drift Detection & Model Monitoring**.